# Colour Brings People Together
### Part C · Creating an empirically grounded colour palette (60 min)

<img src="https://raw.githubusercontent.com/akvilesinkeviciute/colour-brings-people-together/v1.0/images/serandu_example.png" alt="Serandu in OtjiHimba: the AI's estimate, a brown, against speakers' best example, a wine red, ΔE₀₀ 25.4 apart" width="420">

In Part B you found what you and your audience call your test colours. Now go the other way: give those **names** to an AI, ask which colour each name maps to, and check its answers against the Colour Spinner's empirical data. Where the AI is off, replace its colour with the speakers' best example. You finish with a palette for each audience that you can defend with evidence.

Why it matters: in the *Colour in Translation* benchmark (Mylonas et al., CHI 2026), a current large language model's colours were far from speakers' focal colours, because it mapped names onto the same prototypes whatever the language.

Run each cell with **Shift + Enter**. Change only the grey form fields and the menus that appear in some cells.

In [ ]:
#@title Run me first: load the masterclass toolkit
#@markdown Takes about 30 seconds the first time. You're ready when it says **Toolkit ready ✓**.
TOOLKIT_URL = "https://raw.githubusercontent.com/akvilesinkeviciute/colour-brings-people-together/v1.0/colour_masterclass.py"
import os, importlib, urllib.request
try:                                   # always fetch the current version
    urllib.request.urlretrieve(TOOLKIT_URL, "colour_masterclass.py")
except Exception as e:
    if not os.path.exists("colour_masterclass.py"):
        raise RuntimeError("Could not download the toolkit. Check your internet connection, or upload "
                           "colour_masterclass.py in Colab's Files panel (folder icon, left).") from e
    print("Couldn't refresh the toolkit, so using the copy already here.")
import colour_masterclass
importlib.reload(colour_masterclass)
from colour_masterclass import *

## 1 · Your names from Part B

Run the next cell and upload the `partB_results.csv` you downloaded at the end of Part B; the upload button appears when it runs. If you lost the file, choose *Type them in* and type a few colour names per language instead.

In [ ]:
#@title Load your names
NamesFrom = "Upload partB_results.csv"  #@param ["Upload partB_results.csv", "Type them in"]
#@markdown Run this cell and upload your Part B results. *Type them in* only if you lost the file.
Backend = "Colour Spinner API"   # facilitators: switch to "Copied from the browser" only if the Spinner can't be reached
names_first(NamesFrom)                    # only with "Type them in": shows boxes to fill in first
names = load_names(NamesFrom)
LANGUAGES = list(dict.fromkeys(names.language))
show_names(names)

## 2 · Ask your AI for the colour of each name

Use the same AI as in Part B. The prompt asks, for each name, for the colour that speakers would consider its **best example**: its most typical instance.

In [ ]:
#@title The prompt for your AI
lines = "\n".join(numbered_items(names, "colour"))
himba = "\n(OtjiHimba is spoken by the Himba of northern Namibia.)" if "OtjiHimba" in LANGUAGES else ""
PROMPT_C = f"""You are helping with a cross-cultural colour naming study.
For each numbered colour name below, give the colour, as a hex code, that native speakers of that language would consider the best example of the name: its most typical instance.{himba}
If you're not certain, give your best estimate: we will compare it with empirical data.

{lines}

Reply with one line per number, in this format and nothing else:
1. #RRGGBB
2. …"""
show_prompt(PROMPT_C, PASTE, rows=names, kind="colour")

In [ ]:
#@title Read the AI's colours
pal = read_ai_colours(collected_answers(), names)       # the colour codes read from the pasted reply

## 3 · Does the AI's colour mean the same to speakers?

Two checks for every name:

- **Distance:** how far the AI's colour is from the speakers' best example (ΔE₀₀). Above 5, the two read as different colours.
- **Round trip:** what speakers would call the AI's colour. A different name means the AI's colour says something else.

*To check a name in the Colour Spinner yourself, choose its language in the Spinner's menu first: the Spinner opens in American English, and each language has its own best examples.*

In [ ]:
#@title Compare the AI's colours with the speakers' best examples
ReplaceAbove = DELTA_E_THRESHOLD              # ΔE₀₀ 5, the threshold used throughout

show(small("Looking up every name's best example and what speakers call the AI's colours; this takes a minute, please wait…"))
spinner = connect(Backend)
pal = add_best_examples(pal, spinner)                   # the speakers' best example of each name
pal["de00"] = [de00(ai, best) if isinstance(ai, str) and isinstance(best, str) else np.nan
               for ai, best in zip(pal.ai_hex, pal.spinner_hex)]
pal["rt_name"], pal["rt_conf"] = None, np.nan
for k, row in pal.iterrows():                           # round trip: what speakers call the AI's colour
    if isinstance(row.ai_hex, str):
        namer = safe(spinner.name, row.ai_hex, row.language)
        if namer:
            pal.at[k, "rt_name"], pal.at[k, "rt_conf"] = namer["name"], namer["confidence"]
pal = pal.sort_values(["language", "de00"], ascending=[True, False]).reset_index(drop=True)
show_comparison(pal, ReplaceAbove)
error_summary(pal, LANGUAGES, ReplaceAbove)

**Where is the AI off, and in which direction?** Each panel shows the names for one test colour on the CIELAB a*–b* plane, seen from above: the AI's colour (outlined) joined to the speakers' best example (filled). The longer the line, the further the AI is from what speakers mean. Lines that all point the same way suggest a systematic bias, for instance towards English focal colours.

In [ ]:
#@title Where the AI is off
plot_ai_vs_speakers(pal, LANGUAGES, ReplaceAbove)

## 4 · Ground and repair the palette, step by step

Three short steps. Each shows only what it changed, and the last one gives your final palette.

### Step 1 · Ground: use what speakers mean

Wherever the AI's colour is wrong, it's replaced by speakers' best example of the name. Choose the rule: too far from the best example (more than ΔE₀₀ 5), speakers would call the AI's colour something else, or either.

In [ ]:
#@title Step 1 · Ground { run: "auto" }
ReplaceWhen = "Either"  #@param ["Either", "Too far from the best example", "Speakers would use a different name"]
#@markdown **ReplaceWhen**: which check decides that the AI's colour is wrong.
pal_grounded = ground_palette(pal, ReplaceWhen)
show_grounded(pal_grounded, LANGUAGES)

### Step 2 · Give every test colour its own name

Two test colours with the same name (6 and 14, both *burou* in OtjiHimba) would be confused when people talk about them, however different they look. The later one gets a different name: the closest synonym in the Spinner's Thesaurus whose best example can be told apart from your other colours.

How well this works depends on the size of a language's colour vocabulary. With 32 to 47 terms (English, French, Greek, Serbian, Spanish), there's usually a close synonym. With OtjiHimba's seven, the closest names may already be taken, so a renamed colour can get a name that means something rather different: the palette becomes communicable, but at a cost.

In [ ]:
#@title Step 2 · Rename shared names
pal_named = name_palette(pal_grounded, spinner, LANGUAGES)

### Step 3 · Repair for one kind of vision

Choose whose eyes you are correcting for. Where two colours look alike for that viewer (closer than ΔE₀₀ 5), one moves to another colour of the same name in the Spinner's Mapper, the smallest move that separates them. Then you see the closest pair left, and your final palette.

A pair that's still too close can't be separated within its names: give it a second cue, such as a pattern, label or icon. Try another kind of vision to see how the repairs change.

In [ ]:
#@title Step 3 · Repair { run: "auto" }
Vision = "Typical"  #@param ["Typical", "Protanopia", "Deuteranopia", "Tritanopia", "Age 40", "Age 60", "Age 80"]
#@markdown **Vision**: whose eyes the palette is corrected for.
show(small("Checking colours of the same name with the Colour Spinner; this can take a minute, please wait…"))
pal = repair_for(pal_named, spinner, LANGUAGES, Vision)

## 5 · See the palette in use

Choose a language and a kind of design (swatches, a bar, donut, line or scatter chart, or a world map, as in our palette explorer) to see its grounded palette in use, next to how it looks with another kind of vision. **Colours** shows just your test colours, or all 17 SDG colours with your test colours grounded. The values are example data; each preview you make (language, colours, design and vision) is saved for the download.

In [ ]:
#@title Preview
#@markdown Choose a language, which colours, a kind of design and a kind of vision.
live_controls({"language": ("Language", LANGUAGES, my_language(names)),
               "colours":  ("Colours", ["Your test colours", "All 17 SDG colours"], "Your test colours"),
               "design":   ("Design", ["Colour swatches", "Bar chart", "Donut chart", "Line chart", "Scatter plot", "Map (world)"], "Bar chart"),
               "vision":   ("Vision", VISIONS, "Deuteranopia")},
              lambda language, colours, design, vision: preview_applications(pal, language, vision, kind=design, colours=colours,
                  path=f"preview_{language.replace(' ', '_')}_{design.split(' ')[0].lower()}{'_all17' if colours.startswith('All') else ''}_{vision.replace(' ', '_').lower()}.png"))

## 6 · Your grounded colours in the full SDG palette

Back to the UN's full palette: choose a language to see the 17 colours with your test colours replaced by their grounded colours, under any kind of vision. Grounding fixes what the names mean, but it can create new problems elsewhere in the palette. The arcs under the palette show which colours to check again:

- **Red arcs, with an arrow: new problems.** Grounding has moved a test colour closer than ΔE₀₀ 5 to another colour, although the UN's original colours were distinguishable. The arrow points from the grounded colour to the colour it is now close to.
- **Grey dashed arcs: problems that were already there.** These pairs were already closer than ΔE₀₀ 5 in the UN's palette, and still are.

The notes under the arcs give the exact distances. Change **Vision** to see which pairs become close for a colour vision deficiency or an older eye.

*Only your test colours are grounded; testing the whole palette would mean checking all 17 colours together, for every audience. And the SDG colours never work alone: each goal also has a number, an icon and a name, which keep the goals apart. Here the palette is an example: this shows what evidence could improve, not a proposal to change the UN's colours.*

In [ ]:
#@title Grounded colours in the full palette
#@markdown Choose a language and a kind of vision.
live_controls({"language": ("Grounded for", [l for l in LANGUAGES if l in set(pal.language)], my_language(names)),
               "vision":   ("Vision", VISIONS, "Typical")},
              lambda language, vision: show_corrected_sdg(pal, language, vision))

## 7 · Export your palette

Run the next cell to download one zip with what you saw above: your grounded palette by test colour and language (as a picture and a table with the evidence), section 6 for each language with its arcs and notes, all 17 SDG colours for each audience, and the previews you made.

In [ ]:
#@title Download your palette
export_palette(pal, LANGUAGES)

## 8 · Your most striking finding

The card shows the name where the AI's colour was furthest from what speakers mean; type another name to choose a different one. Talk it through with your partner, or on your own note what it shows, and bring it to the plenary; the optional questionnaire asks about it too.

In [ ]:
#@title Finding to share { run: "auto" }
FindingName = ""  #@param {type:"string"}
#@markdown Leave empty for the name where the AI was furthest from speakers, or type another name.
show_finding(pal, FindingName)

## 9 · Tell us how it went (optional)

In [ ]:
#@title The questionnaire
show_questionnaire()

---
#### Credits and references

**Colour naming data and simulations:** the Colour Spinner, [colornaming.net](https://colornaming.net); Sinkeviciute, Ahmed, Stutters, Koliousis & Mylonas (2026), under review. Colour vision deficiencies: Viénot, Brettel & Mollon (1999); ageing: Pokorny, Smith & Lutze (1987).

**AI colour-naming benchmark:** Mylonas, Ahmed, Sinkeviciute & Koliousis, *Colour in Translation*, CHI 2026.

**The *serandu* example:** the AI's estimate from ChatGPT, asked on 26 October 2026; the speakers' best example from the Colour Spinner.

**SDG colours:** United Nations, [Sustainable Development Goals communications materials](https://www.un.org/sustainabledevelopment/news/communications-material/); [un.org/sustainabledevelopment](https://www.un.org/sustainabledevelopment). *The content of this publication has not been approved by the United Nations and does not reflect the views of the United Nations or its officials or Member States.*

**Maps:** world map from Natural Earth (public domain) and US states map from the US Census Bureau, both via vega-datasets; example values only.

**Colour science:** CIELAB and CIEDE2000 with the [colour-science](https://www.colour-science.org) library.